In [1]:
import pandas as pd
import numpy as np

In [2]:
VocabISOT = pd.read_csv("../vocabulary/VOCAB_ISOT.csv")
NewsISOT = pd.read_csv("../data/processed/NewsISOT.csv")
word2id = dict(zip(VocabISOT["word"], VocabISOT["id"]))

In [3]:
id2word = {idx: word for word, idx in word2id.items()}

In [4]:
import re

def preprocess_words(word):

    # Convert to lowercase
    word = word.lower()

    # Remove the punctuations, signs at the start or end of words
    word = re.sub(r"^[.,:;!?(){}\[\]\"'`]+|[.,:;!?(){}\[\]\"'`]+$","",word)

    # Splitting the numeric, alphabetical and - separated parts
    return re.findall(r"[@#$%]|[a-z']+|\d+",word)
    

In [5]:
SPECIAL = {
    "#": "<#>",
    "@": "<@>",
    "$": "<$>",
    "%": "<%>"
}

def normalize_word(token):

    if re.match(r'^(https?://|www\.)',token.lower()):
        return ["<LINK>"]

    normalized = []
    
    for piece in preprocess_words(token):
    
        if piece in SPECIAL:
            normalized.append(SPECIAL[piece])
    
        elif piece.isdigit():
            n = len(piece)
    
            if n == 1:
                normalized.append("<NUM1>")
            elif n == 2:
                normalized.append("<NUM2>")
            elif n == 3:
                normalized.append("<NUM3>")
            elif n == 4:
                normalized.append("<NUM4>")
            else :
                normalized.append("<NUM4+>")
    
        else:
            normalized.append(piece)

    return normalized  

In [6]:
word2id = dict(zip(VocabISOT["word"], VocabISOT["id"]))

In [7]:
from collections import Counter
from scipy.sparse import csr_matrix
from tqdm.auto import tqdm

In [8]:
def article_to_sparse(row, word2id):

    counts = Counter()

    for column in ["title", "text"]:

        if pd.isna(row[column]):
            continue

        for token in row[column].split():
            counts.update(normalize_word(token))
        
    unk = word2id["<UNK>"]

    indices = []
    data = []

    for word, freq in counts.items():
        idx = word2id.get(word, unk)
        indices.append(idx)
        data.append(freq)

    return indices, data

In [9]:
data = []
indices = []
indptr = [0]

labels = []

encodedVectors = []
label = []

for i in tqdm(range(len(NewsISOT)), desc="Encoding articles"):

    row = NewsISOT.iloc[i]
    
    cols, vals = article_to_sparse(row, word2id)

    indices.extend(cols)
    data.extend(vals)

    indptr.append(len(indices))

    labels.append(row["authenticity"])

Encoding articles:   0%|          | 0/38214 [00:00<?, ?it/s]

In [10]:
X = csr_matrix(
    (data, indices, indptr),
    shape=(len(NewsISOT), len(word2id)),
    dtype = np.int32
)

y = np.array(labels, dtype=np.int8)

In [11]:
print(X.shape)
print(X.nnz)
print(y.shape)

(38214, 21606)
8127079
(38214,)


In [13]:
# from scipy.sparse import save_npz

# save_npz("../encoding/X_ISOT.npz", X)
# np.save("../encoding/y.npy", y)

In [1]:
# To reload the data

# from scipy.sparse import load_npz

# X = load_npz("../encoding/X_ISOT.npz")
# y = np.load("../encoding/y.npy")